# user 메시지 여러 개를 한 요청에 묶어 보내기

Claude Code 소스를 보면 한 요청에 user 메시지를 2~3개씩 묶어 보내는 패턴이 자주 나옵니다.
(system-reminder 주입, 파일 내용 전달, 실제 사용자 입력을 **각각 별도의 user 메시지**로 분리)

OpenAI에서도 똑같이 가능합니다 — Responses API의 `input`에 **메시지 배열**을 넣으면 되고,
같은 `user` 역할 메시지를 연속으로 여러 개 넣는 것도 허용됩니다.

이렇게 분리하는 이유:
1. **출처 분리** — 시스템이 주입한 컨텍스트 vs 진짜 사용자 입력을 구조적으로 구분 (문자열 이어붙이기보다 명확)
2. **캐싱 효율** — 고정 컨텍스트를 앞 메시지로, 변하는 질문을 뒤 메시지로 두면 프리픽스 캐시에 유리
3. **일괄 전달** — 큐에 쌓인 여러 입력을 한 번의 API 호출로 처리

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-5-nano"

## 1. 기본 — user 메시지 3개를 한 번에

연속된 user 메시지는 **모두 컨텍스트로 읽히지만**, 모델은 **마지막 메시지를 '현재 요청'으로 취급**하는 경향이 있습니다.
(실제로 질문 3개를 나란히 보내면 마지막 질문에만 답하는 경우가 많습니다)

그래서 마지막 메시지에서 앞 메시지들을 참조하게 하면, 모든 메시지가 전달됐는지 확인할 수 있습니다.

In [2]:
response = client.responses.create(
    model=MODEL,
    input=[
        {"role": "user", "content": "질문 A: 파이썬 리스트 컴프리헨션이 뭐야?"},
        {"role": "user", "content": "질문 B: 제너레이터랑 차이는?"},
        {"role": "user", "content": "위의 질문 A, B에 각각 한 문장으로 답해줘."},
    ],
)

print(response.output_text)

A: 파이썬 리스트 컴프리헨션은 반복문과 조건문을 한 줄로 써서 새로운 리스트를 간결하게 만드는 문법입니다.

B: 리스트 컴프리헨션은 결과를 즉시 리스트로 모두 만들지만, 제너레이터는 값을 필요할 때 하나씩 생성해 메모리를 더 적게 사용합니다.


## 2. 실전 패턴 — Claude Code 스타일 컨텍스트 분리

Claude Code가 실제로 쓰는 구조를 재현합니다. 한 요청 안에 성격이 다른 user 메시지 3개를 쌓습니다:

| 순서 | 메시지 성격 | Claude Code에서의 예 |
|---|---|---|
| 1 | 시스템이 주입한 리마인더/컨텍스트 | `<system-reminder>` 블록 |
| 2 | 참고 자료 (파일 내용 등) | 열어둔 파일, diff |
| 3 | 진짜 사용자 입력 | 사용자가 타이핑한 질문 |

고정적인 것(1, 2)을 앞에, 변하는 것(3)을 뒤에 두는 배치가 프롬프트 캐싱과도 궁합이 좋습니다.

In [3]:
context_reminder = (
    "<system-reminder>\n"
    "당신은 코드 리뷰 도우미입니다. 답변은 한국어로, 3문장 이내로 간결하게 하세요.\n"
    "버그를 발견하면 반드시 수정 코드를 함께 제시하세요.\n"
    "</system-reminder>"
)

file_content = (
    "파일: utils.py\n"
    "```python\n"
    "def add_item(item, items=[]):\n"
    "    items.append(item)\n"
    "    return items\n"
    "```"
)

user_question = "이 함수 뭔가 이상하게 동작하는데 버그 있어?"

response = client.responses.create(
    model=MODEL,
    input=[
        {"role": "user", "content": context_reminder},   # 1) 주입된 컨텍스트
        {"role": "user", "content": file_content},        # 2) 참고 자료
        {"role": "user", "content": user_question},       # 3) 진짜 사용자 입력
    ],
)

print(response.output_text)

네, `items=[]`가 함수 정의 시 한 번만 생성되어 호출 간에 리스트가 공유되는 버그가 있습니다.  

```python
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items
```

이렇게 하면 인자를 생략할 때마다 새 리스트를 사용합니다.


## 3. developer 메시지와 조합

역할 우선순위는 `developer`(= 구 system) > `user`입니다.
고정 규칙은 developer 메시지에, 컨텍스트와 질문은 user 메시지 여러 개에 나눠 담는 조합이 일반적입니다.

In [4]:
response = client.responses.create(
    model=MODEL,
    input=[
        {"role": "developer", "content": "모든 답변은 반말 금지, 존댓말로. 답변 끝에 항상 '✅ 검토 완료'를 붙이세요."},
        {"role": "user", "content": "[참고] 우리 팀 컨벤션: 함수 이름은 snake_case를 쓴다."},
        {"role": "user", "content": "함수 이름을 getUserName으로 지어도 돼?"},
    ],
)

print(response.output_text)

팀 컨벤션이 함수 이름은 `snake_case`라고 되어 있으므로 `getUserName`처럼 `camelCase`로 짓는 것은 피하시는 게 좋습니다.

대신 아래처럼 작성하시면 됩니다.

```python
get_user_name
```

즉, 함수 이름은 `getUserName`이 아니라 `get_user_name`으로 지으시는 것을 권장드립니다. ✅ 검토 완료


## 4. 대안 — 한 user 메시지 안에 content 파트 여러 개

메시지를 나누는 대신, **하나의 user 메시지** 안에 `input_text` 파트를 여러 개 넣을 수도 있습니다.
이미지·파일과 텍스트를 섞을 때 쓰는 형식과 동일합니다.

- **메시지 분리** (1~3번 방식): 대화 기록상 별개의 턴처럼 구조화 — 출처가 다른 내용을 구분할 때
- **파트 분리** (이 방식): 논리적으로 한 발화지만 내용물이 여러 개일 때 (텍스트 + 이미지 등)

In [5]:
response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "다음 두 에러 로그를 보고 공통 원인을 한 문장으로 추정해줘."},
                {"type": "input_text", "text": "[로그 1] ConnectionResetError: [Errno 54] Connection reset by peer"},
                {"type": "input_text", "text": "[로그 2] requests.exceptions.ReadTimeout: HTTPSConnectionPool(host='api.example.com', port=443): Read timed out."},
            ],
        }
    ],
)

print(response.output_text)

공통 원인은 `api.example.com` 서버 또는 중간 네트워크/프록시의 불안정으로 인해 연결이 강제로 끊기거나 응답이 지연된 것으로 추정됩니다.


## 정리

- Responses API의 `input`은 메시지 배열을 받고, **연속된 user 메시지 여러 개**를 그대로 허용
- Claude Code처럼 "주입 컨텍스트 / 참고 자료 / 실제 입력"을 별도 메시지로 분리하면 프롬프트 관리가 깔끔해지고 캐싱에도 유리
- 역할 우선순위: `developer`(system) > `user` — 규칙은 developer에, 내용은 user에
- 한 발화 안의 복수 내용물은 `content` 파트 배열(`input_text` 등)로도 표현 가능
- 참고: 구형 Chat Completions API(`messages=[...]`)에서도 연속 user 메시지는 동일하게 허용됩니다